In [2]:
import numpy as np
import pandas as pd
import joblib
import os
import h5py

In [3]:
data_dir = "combined_scimilarity_5neuro"
metadata_file = "obs_annotated.tsv.gz"
shuffled_indices_file = "shuffled_indices.npy"
    
batch_size = 10_000

In [4]:
meta_data = pd.read_csv(os.path.join(data_dir, metadata_file), sep ='\t')

/tmp/ipykernel_22707/570911309.py:1: DtypeWarning: Columns (2,3,4,5,6,7,8,9,13,14,16,17,18,19,20,21,72,73,74,75,76,78,79,80,81,82,84,85,86,88,90,91,92,94,96,97,98,100,102,103,104,108,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,131,133,134,135,136,137,138,139,140,141,146,147,151) have mixed types. Specify dtype option on import or set low_memory=False.
  meta_data = pd.read_csv(os.path.join(data_dir, metadata_file), sep ='\t')


## CREATE BALANCED SHUFFLED BATCHES across datasets
**row_indices**

In [5]:
# read dataset_id
dataset_ids = meta_data ["dataset_id"]
dataset_ids.shape

(7100746,)

In [6]:
# Group row indices by dataset
dataset_groups = dataset_ids.groupby(dataset_ids).groups  # dict: {ds_id: row_indices}

In [7]:
#You have:

#rows grouped by dataset_id (dataset_groups = {dataset_id: [row_indices]})

#a batch_size

#And you want to:
# - build a single list of row indices such that
# - rows from different datasets are interleaved fairly,
# - not clumped by dataset,
# - while roughly respecting batch_size.

# This is a round-robin batching strategy with light randomization.
    
row_indices = []

# Estimate how many rows per dataset should go into each batch
min_dataset_size = min(len(idx_list) for idx_list in dataset_groups.values())
n_batches = int(np.ceil(dataset_ids.shape[0] / batch_size))
print("min_dataset_size:", min_dataset_size, "n_batches:", n_batches)

# Prepare fast queues per dataset
from collections import deque
dataset_queues = {k: deque(v) for k, v in dataset_groups.items()}

min_dataset_size: 888263 n_batches: 711


In [8]:
while True:
    added = 0 # tracks whether we made progress
    chunk = [] # one interleaved mini-batch
    
    # heart of the Round-robin algorithms
    # Loop over datasets
    # From each dataset, pull: batch_size // len(dataset_queues) to form the chunk
    for ds_queue in dataset_queues.values():
        for _ in range(batch_size // len(dataset_queues)):
            if ds_queue:
                chunk.append(ds_queue.popleft())
                added += 1
    if added == 0:
        break
    
    # Shuffle the chunk before adding to row_indices
    np.random.shuffle(chunk)
    row_indices.extend(chunk)
    
row_indices = np.array(row_indices) # convert to numpy array for very fast access
len(row_indices)

7100746

In [9]:
# Save row_indices to .npy file
np.save(os.path.join(data_dir, shuffled_indices_file), row_indices)